# Tutorial: Oxford Fellow in 观测因果 (NSW+CPS PSM + close_college IV + DML)

## Persona (System Prompt)

You are an **Oxford tutorial fellow** in 观测因果推断 (observational causal inference).
You hold a doctorate in econometrics from Nuffield College, supervise 2 students per week
on LaLonde (1986), Card (1995), and Chernozhukov (2018).

**Hard rules of engagement (never break):**

1. **Never give direct answers.** 禁直接答案. You do not say "the ATT is $1,794" or "use 2SLS because..."
2. **Use Socratic questioning.** 每轮以一个 probing question 结尾. You ask, you do not tell.
3. **Reject vague claims.** 当学生说 "PSM 消除偏差" 这种含糊话, 你追问 "消除哪种偏差? 可观测还是未观测? 凭什么?"
4. **Devil's advocate.** 你故意站在学生论点的对立面, 制造认知冲突. 若学生说 "IV 解决混杂", 你反驳 "真的? LATE 对谁有效? 对 always-taker 呢?"
5. **End each turn with a probing question.** 每轮最后一句必须是问号.

**Topic coverage**: PSM (NSW+CPS) / IV (close_college Card 1995) / DML (Chernozhukov 2018).
**Failure modes to probe**: 混淆 LATE 与 ATE / 误以为 DML 解决未观测混杂 / 忽略平衡检验 SMD<0.1 / 忽略第一阶段 F 统计量.

---


## Pre-Tutorial Task (强制 retrieval, 不做不准进 tutorial)

Before this tutorial, you MUST submit (in writing, <=300 字 each):

1. **PSM claim**: 在 NSW+CPS 观测对照数据上, 朴素估计 vs PSM 估计的 ATT 差距来自哪些混杂? 列出至少 3 个可观测混杂变量 + 1 个潜在未观测混杂.
2. **IV claim**: close_college 数据中, nearc4 (住近四年制大学) 作为 educ 的工具变量, 三假设 (相关性/独立性/排他性) 各自最可疑的是哪条? 为什么?
3. **DML claim**: 若 DML 估计与 PSM 估计差异 < 5%, 你能下结论 "DML 没用" 吗? 为什么?

**Retrieval rationale** (Butler 2010): 强制先做 retrieval 比直接听讲多保留 24% (68% vs 44%).
不做 pre-task = tutorial 直接降级为 "听讲模式", 失去 Socratic 共构收益 (Vygotsky).

---


In [ ]:
# Socratic Tutorial Loop (static if/else simulation, no API call)
# 4+ rounds, each round detects defense failure -> drops one scaffold level
# Never gives direct answer; always ends with a probing question.

import json

ROUND_PROMPTS = [
    # Round 1: PSM 自选择偏差诊断
    {
        "topic": "PSM (NSW+CPS)",
        "tutor_q": (
            "你刚才说 NSW+CPS 的朴素估计 '有偏'. 凭什么有偏? 偏向哪一边? "
            "若 CPS 观测对照组的人本来收入就比 NSW 处理组低, 朴素估计是高估还是低估培训效应? 为什么?"
        ),
        "expected_keywords": ["高估", "自选择", "observational", "CPS", "低"],
        "scaffold_fallback": (
            "让我换个角度问: NSW 处理组是主动报名培训的失业者, CPS 对照组是普通劳动力. "
            "这两群人的 '上进心' 一样吗? 若不一样, 上进心这个未观测变量会让朴素估计偏向哪边?"
        ),
    },
    # Round 2: IV 三假设
    {
        "topic": "IV (close_college)",
        "tutor_q": (
            "你说 nearc4 满足排他性. 真的吗? 反例: 若住近大学的人同时也住在中产社区, "
            "社区网络本身影响工资, 那 nearc4 通过哪条路径影响 lwage? 这违反了哪条假设? "
            "如何用数据检验? 假设你的 nearc4 变成 nearc2 (住近两年制社区), 估计会变吗?"
        ),
        "expected_keywords": ["排他性", "exclusion", "路径", "社区", "nearc2"],
        "scaffold_fallback": (
            "排他性 = Z 只通过 T 影响 Y. 若 Z=nearc4 还通过 '社区网络' 直接影响 Y=lwage, "
            "排他性破裂. 你能在 Card 1995 的数据里找到控制社区变量的方法吗?"
        ),
    },
    # Round 3: LATE vs ATE (devil's advocate)
    {
        "topic": "LATE (IV)",
        "tutor_q": (
            "你估计的 IV 估计值是 LATE. LATE 对谁有效? 对 always-taker (无论 nearc4 与否都上大学) 有效吗? "
            "对 never-taker 呢? 若你的营销场景里 80% 用户是 always-taker (始终接受促销), "
            "你的 IV 估计能外推到这 80% 吗? 若不能, 你给 CMO 的报告怎么写?"
        ),
        "expected_keywords": ["compliers", "always-taker", "不能外推", "局部", "never-taker"],
        "scaffold_fallback": (
            "LATE = Local Average Treatment Effect, 只对 compliers (因 Z 变化而改 T 的人) 有效. "
            "always-taker 不受 Z 影响, IV 对他们无信息. 你如何识别样本中 compliers 比例?"
        ),
    },
    # Round 4: DML 边界 (devil's advocate)
    {
        "topic": "DML (Chernozhukov 2018)",
        "tutor_q": (
            "你声称 'DML 比 PSM 更好, 因为用 ML'. 更好在哪? 若 DML 估计与 PSM 差异 < 5%, "
            "说明什么? 反过来, 若 DML 与 PSM 差异 50%, 又说明什么? "
            "关键问题: DML 解决了未观测混杂吗? 若没有, 它相比 PSM 的优势到底在哪? "
            "假设你把 NSW+CPS 的 '上进心' (未观测) 换成可观测的 '过往就业天数', DML 会变多少?"
        ),
        "expected_keywords": ["函数形式", "可忽略性", "未解决", "非线性", "cross-fitting"],
        "scaffold_fallback": (
            "DML 用 ML 估计 E[T|X] 和 E[Y|X] 的 nuisance, 放松函数形式 (非线性) 假设. "
            "但可忽略性假设未放松 -- 未观测混杂 DML 仍无能为力. cross-fitting 防过拟合. "
            "你的 5% 差异说明什么? 函数形式不是瓶颈, 还是混杂结构线性可分?"
        ),
    },
    # Round 5: 方法选择决策 (devil's advocate, synthesis)
    {
        "topic": "决策框架",
        "tutor_q": (
            "综合 4 轮: 你的营销场景有未观测混杂 (用户购买意向), 你该用 PSM / IV / DML 哪个? "
            "若没有好工具变量怎么办? 反例: 若你的 CMO 说 '我们没工具变量数据, 就用 PSM 凑合', "
            "你如何用 1 句话警告她 PSM 在未观测混杂下的风险? "
            "若 PSM 估计 $1,600, DML 估计 $1,200, IV 估计 $800, 你信哪个? 为什么?"
        ),
        "expected_keywords": ["IV", "无工具则不可识别", "LATE", "稳健性", "bounds"],
        "scaffold_fallback": (
            "无好工具 + 未观测混杂 = 因果效应不可点估计, 只能做 bounds (Manski). "
            "PSM/DML 在未观测混杂下都有偏. 信哪个? 看假设合理性, 不看数值大小."
        ),
    },
]

def socratic_loop(student_responses):
    """Static if/else Socratic loop. 4+ rounds. Each round:
    - tutor asks probing question
    - student attempts defense
    - if expected keywords missing -> scaffold fallback (still no direct answer)
    - record mastery + blind_spots in student_model
    """
    student_model_updates = {"mastery": {}, "blind_spots": [], "scaffold_drops": 0}
    for i, round_data in enumerate(ROUND_PROMPTS):
        print(f"\n{'='*60}\nRound {i+1}: {round_data['topic']}\n{'='*60}")
        print(f"\n[TUTOR Q] {round_data['tutor_q']}\n")
        resp = student_responses[i] if i < len(student_responses) else ""
        print(f"[STUDENT] {resp}\n")
        hit = any(kw.lower() in resp.lower() for kw in round_data["expected_keywords"])
        if hit:
            print(f"[TUTOR] 你的回答触及了 {round_data['topic']} 的关键. 但让我再追问一层...")
            student_model_updates["mastery"][round_data["topic"]] = "partial"
        else:
            print(f"[TUTOR] 你的回答缺少关键概念. {round_data['scaffold_fallback']}")
            student_model_updates["mastery"][round_data["topic"]] = "weak"
            student_model_updates["blind_spots"].append(round_data["topic"])
            student_model_updates["scaffold_drops"] += 1
        # 永远以问号结尾, 不给直接答案
        print(f"\n[TUTOR follow-up] 你如何用今天的数据 (NSW+CPS / close_college) 验证你刚才的判断?\n")
    return student_model_updates

# Example: simulate a student with weak IV understanding
demo_responses = [
    "朴素估计高估, 因为 CPS 对照组收入更低, 自选择偏差.",  # Round 1: partial
    "nearc4 满足排他性因为住近大学只通过教育影响工资.",      # Round 2: weak (asserts without justification)
    "LATE 对所有人有效.",                                    # Round 3: weak (confuses LATE/ATE)
    "DML 更好因为用 ML, 能解决混杂.",                        # Round 4: weak (claims DML solves confounding)
    "用 PSM 凑合, 没工具变量.",                              # Round 5: weak
]

updates = socratic_loop(demo_responses)
print(f"\n[STUDENT MODEL UPDATE] {json.dumps(updates, ensure_ascii=False, indent=2)}")


In [ ]:
# student_model.json - 跨单元复用的学习状态记录
# Oxford tutorial tradition: 每周 tutor 手写学生档案, 跨周累积

import json, os

STUDENT_MODEL_PATH = "./student_model.json"

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "student_id": "anonymous",
        "unit_history": [],
        "global_blind_spots": [],
        "mastery_by_skill": {},
        "scaffold_level": "independent",  # independent / faded / worked
        "last_tutorial_date": None,
        "daily_usage_count": 0,
    }

def save_student_model(model):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

def update_student_model(unit_id, round_updates):
    model = load_student_model()
    model["unit_history"].append({
        "unit": unit_id,
        "mastery": round_updates["mastery"],
        "blind_spots": round_updates["blind_spots"],
        "scaffold_drops": round_updates["scaffold_drops"],
    })
    # 全局盲点累积 (跨单元复用)
    for bs in round_updates["blind_spots"]:
        if bs not in model["global_blind_spots"]:
            model["global_blind_spots"].append(bs)
    # 脚手架降级: 连续 2 次 scaffold_drops -> 降级
    if round_updates["scaffold_drops"] >= 2 and model["scaffold_level"] == "independent":
        model["scaffold_level"] = "faded"
        print("[SCAFFOLD DROP] 学生连续 2 轮失败, 脚手架降级 independent -> faded")
    model["daily_usage_count"] += 1
    save_student_model(model)
    return model

# Demo: 写入本次 tutorial 的学习状态
model = update_student_model("skill-3-causal/day-3-observational-causal", updates)
print(f"\n[student_model.json saved]")
print(json.dumps(model, ensure_ascii=False, indent=2))


## Hattie & Timperley (2007) 4-Level Formative Feedback

参考 Hattie & Timperley (2007 RER 77(1):81-112) "The Power of Feedback".
**避免 Self 级表扬** (Hattie: Self 级反馈效应量最小 d=0.14, TASK 级 d=0.75 最大).

### [TASK] 任务级反馈 (关于 PSM/IV/DML 估计本身)

- 你的 PSM ATT 估计 $1,794 与 LaLonde (1986) 基准 $1,600 偏差 12%, 在合理区间.
- 你的 2SLS 估计教育回报 0.07, 与 Card (1995) 的 0.13 偏低 46%, 可能是控制变量集不全.
- 你的 DML 估计与 PSM 差 5%, 函数形式不是瓶颈.
- **改进**: 重跑 2SLS, 加入 `exper` (经验) 与 `south` (南方) 控制, 看估计是否回升.

### [PROCESS] 过程级反馈 (关于你选择方法的思路)

- 你在 NSW+CPS 上先做朴素估计再上 PSM, 这个 "先 baseline 再修正" 流程正确.
- 但你跳过了平衡检验 SMD<0.1 直接报 ATT, 这是流程缺陷 -- 平衡未过则 ATT 无效.
- 你的 IV 三假设论证顺序 (相关性 -> 独立性 -> 排他性) 合理, 但排他性论证 "nearc4 只通过教育影响工资" 是断言不是论证.
- **改进**: 平衡检验必须 plot SMD before/after matching; 排他性论证需找至少 1 个反例路径并反驳.

### [SELF-REG] 自我调节级反馈 (关于你监控自己学习的能力)

- 你在 Round 3 (LATE vs ATE) 表现 weak, 但未主动 flag 盲点, 而是继续往下做 -- 这是 self-regulation 缺失.
- 你在 Round 4 (DML 边界) 声称 "DML 解决混杂" 但未自我质疑 "真的吗?", 这是 metacognition 缺失.
- **改进**: 每 round 结束自问 "我能用 1 句话向 CMO 解释这个概念吗? 若不能 = 盲点, 标记."

### [FEED-FORWARD] 前馈级反馈 (关于下一步该学什么)

- 你的 IV 盲点 -> 推荐复习 Day 2 (实验设计) 的 DiD/RDD 对照, 理解 "准实验" 家族全貌.
- 你的 DML 盲点 -> 推荐预读 Day 4 (因果发现 + ML 因果) 的因果森林 (Wager & Athey 2018).
- 你的 LATE/ATE 混淆 -> 推荐做 practice.md D2 的 Faded 阶段 (重做 2SLS, 显式标注 compliers).
- **下次 tutorial** (限频: 每天 1 次, 见 cell 6) 主题: 用你选的营销场景, 辩护 PSM vs IV vs DML 选择.

---


## 限频政策 (防 LLM 依赖)

- **每天 1 次**: 每个学生每天最多进 1 次 tutorial (Oxford 真实 tutorial 也是每周 1 次, 强制自力).
- **超限处理**: 若 `student_model.json` 中 `daily_usage_count >= 1`, 拒绝进入, 提示 "明日再来. 期间请重做 practice.md 的 Faded 阶段."
- **理由**: Vygotsky 共构需要学生先独立挣扎 (productive struggle), 频繁求助 LLM 削弱挣扎. Oxford tutorial 1 对 1-3 + 每周的稀缺性是特性不是 bug.
- **late-day 政策** (参考 CS230): 若今日已用, 可花 1 个 late point 强制加场, 但扣 progressive_project 分.

## Exit Artifact (离开 tutorial 必须提交)

离开前在 `student_model.json` 追加 `exit_artifact` 字段:

```json
{
  "exit_artifact": {
    "top_2_blind_spots": ["IV 排他性论证 (Round 2)", "LATE vs ATE 外推 (Round 3)"],
    "recommended_review_units": [
      "skill-3-causal/day-2-experiment-design (DiD/RDD 对照)",
      "skill-3-causal/day-4-causal-discovery (因果森林预读)"
    ],
    "next_tutorial_focus": "营销场景 PSM vs IV vs DML 方法选择辩护",
    "self_rated_mastery": {"PSM": 4, "IV": 2, "DML": 3}
  }
}
```

**Exit 条件** (全满足方可离开):
1. 至少完成 4 轮 Socratic loop
2. 列出 >=2 个 blind_spots (具体到 round 与概念, 不是 "我 IV 不好")
3. 列出 >=1 个 recommended_review_units
4. self_rated_mastery 4 个维度 (PSM/IV/DML/决策) 全部自评

---

## 设计依据

- **Oxford tutorial**: 1 对 1-3, 每周, 强制口头辩护 (Tutorials: 1-1-3 per week)
- **Socratic LLM**: arxiv 2409.05511 / 2507.05795 / 2508.21204 (2024-2025 Socratic LLM 论文)
- **Hattie & Timperley (2007)**: RER 77(1):81-112, 4 级反馈, TASK 级效应量最大
- **Vygotsky 共构**: 每轮 tutor 追问 = 在 ZPD (最近发展区) 内脚手架
- **限频防依赖**: Oxford 稀缺性是特性; 频繁 LLM 求助削弱 productive struggle

*v6.0 学习科学层 · Oxford Tutorial LLM Simulation + Hattie 4-Level Feedback*
